In [7]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

### Affinity translator

In [13]:
import numpy as np

# Simulating real fMRI data
# 50 brain regions, monitored over 200 time points
# Some regions will intentionally "flash together"
np.random.seed(42)

# Generates random noise
brain_signals = np.random.randn(50, 200)

# Creates an artificial "Functional Network":
# We force Regions 0, 1, 2 and 3 to be identical (Strong Cluster)
brain_signals[1] = brain_signals[0] + np.random.normal(0, 0.1, 200)
brain_signals[2] = brain_signals[0] + np.random.normal(0, 0.1, 200)

In [14]:
from interface.modeling_logic.translators import AffinityTranslator

# Initializes the Translator
# threshold=0.5: Ignores weak correlations
# regularization=0.5: Charges a fee of 0.5 energy to connect a node
translator = AffinityTranslator(threshold=0.5, regularization=0.5)

# Translate to Matrix Q
print("Translating brain signals to QUBO...")
q_matrix_rust = translator.from_time_series(brain_signals)

# Visualizing the first data to check
print("\n--- Excerpt from Matrix Q (First 4x4) ---")
for row in q_matrix_rust[:4]:
    print([f"{x:.2f}" for x in row[:4]])

print(f"\nMatrix ready to send to solver! Size: {len(q_matrix_rust)}x{len(q_matrix_rust)}")

Translating brain signals to QUBO...

--- Excerpt from Matrix Q (First 4x4) ---
['0.50', '-0.99', '-0.99', '0.00']
['-0.99', '0.50', '-0.99', '0.00']
['-0.99', '-0.99', '0.50', '0.00']
['0.00', '0.00', '0.00', '0.50']

Matrix ready to send to solver! Size: 50x50


In [15]:
import torch

from interface.models.gnn import GNNBase

print("Running GNN to predict solution structure...")
model = GNNBase(len(q_matrix_rust))

# Simulated input (node ​​features)
input_features = torch.randn(1, len(q_matrix_rust))

with torch.no_grad():
    gnn_output = model(input_features)
    
priors_list = gnn_output.squeeze().tolist()

Running GNN to predict solution structure...


In [17]:
import core_qyra_solver
sol = core_qyra_solver.solve(q_matrix_rust, priors_list, 1000)
print(sol)

(b'\x01\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', -4.456584473489307)


Optimizing...
Optimization completed.
